In [0]:
!pip install yfinance certifi lxml

In [0]:
import yfinance as yf
import pandas as pd
import requests
import datetime
import time

## II. Retrieve the List of Stock Symbols

In [0]:
url = dbutils.secrets.get(scope="Capstone", key="sandp500url")
tables = pd.read_html(url)
sp500_table = tables[0]
sp500_symbols = sp500_table['Symbol'].tolist()
#display(sp500_table['Symbol'])

In [0]:
url

## III. Retrieve the Company Information Bronze Data and Store into the DBFS

In [0]:
api_key = dbutils.secrets.get(scope="Capstone", key="ModelingPrepAPIKey")
all_data = []

for symbol in sp500_symbols:
    url = f"https://financialmodelingprep.com/api/v3/profile/{symbol}?apikey={api_key}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        if data and isinstance(data, list):
            df = pd.DataFrame(data)
            all_data.append(df)
            print(f"Retrieved data for {symbol}")
        else:
            print(f"No profile data for {symbol}")
        
        time.sleep(0.2)
        
    except Exception as e:
        print(f"Error retrieving data for {symbol}: {e}")
if all_data:
    combined_historical_data_df = pd.concat(all_data, ignore_index=True)
    print("Data combined successfully")
else:
    print("No data retrieved.")

In [0]:
display(combined_historical_data_df)

In [0]:
combined_historical_data_df.to_csv('/dbfs/FileStore/Bronze/Company_Information_Bronze.csv', index=False)

jdbc_url = dbutils.secrets.get(scope="Capstone", key="DatabasejdbcUrlBackup")#DatabasejdbcUrl

connection_properties = {
    "user": dbutils.secrets.get(scope="Capstone", key="DatabaseUsername"),
    "password": dbutils.secrets.get(scope="Capstone", key="DatabasePassword"),
    "driver": dbutils.secrets.get(scope="Capstone", key="DatabaseDriver")
}

In [0]:
old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Bronze.Company_Information",
    properties=connection_properties
)

new_data = spark.read.csv(
    "/FileStore/Bronze/Company_Information_Bronze.csv",
    header=True,
    inferSchema=True
)

In [0]:
display(new_data)

In [0]:
df_old_filtered = new_data.join(old_data.select("symbol"), on="symbol", how="left_anti")

df_spark_Company_Information = df_old_filtered.unionByName(old_data)

df_spark_Company_Information.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Bronze.Company_Information") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("overwrite") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")

In [0]:
# %sql
# CREATE TABLE Company_Information (
#     symbol VARCHAR(10) NOT NULL,
#     price DECIMAL(18,4),
#     beta DECIMAL(18,6),
#     volAvg BIGINT,
#     mktCap BIGINT,
#     lastDiv DECIMAL(18,4),
#     [range] VARCHAR(50),
#     changes DECIMAL(18,4),
#     companyName VARCHAR(255) NOT NULL,
#     currency VARCHAR(10),
#     cik VARCHAR(20),
#     isin VARCHAR(20),
#     cusip VARCHAR(20),
#     exchange VARCHAR(100),
#     exchangeShortName VARCHAR(50),
#     industry VARCHAR(100),
#     website VARCHAR(255),
#     [description] TEXT,
#     ceo VARCHAR(100),
#     sector VARCHAR(100),
#     country VARCHAR(50),
#     fullTimeEmployees INT,
#     phone VARCHAR(20),
#     address VARCHAR(255),
#     city VARCHAR(100),
#     state VARCHAR(50),
#     zip VARCHAR(20),
#     dcfDiff DECIMAL(18,4),
#     dcf DECIMAL(18,4),
#     image VARCHAR(255),
#     ipoDate DATE,
#     defaultImage BIT,
#     isEtf BIT,
#     isActivelyTrading BIT,
#     isAdr BIT,
#     isFund BIT,
#     CONSTRAINT PK_Company_Information PRIMARY KEY (symbol)
# );